# 실험 3 시운전: 실기기 `Session` 모드에서 PADO(`pado_safe`) parameter-shift 2 step 연속 실행

목적: `quantum/train_qpu.ipynb`와 동일한 `Session` + `SamplerV2` 설정으로 PADO(`pado_safe`, 사전학습·증류된 회로 61개 각도 → step당 회로 123개)를 **2 step 연속 실행**하여 다음을 확인한다.
1. **Session 연속 실행 검증** — 첫 번째 job(step 1)이 큐 대기(`QUEUED`)를 거쳐 한 번 실행(`RUNNING`)에 들어가 세션이 활성화되면, 직후 제출되는 두 번째 job(step 2)이 **대기열 없이 즉시 실행(`queue_s ≈ 0초`)**되는지 확인한다.
2. **시간 및 과금(usage)** — step 1과 step 2 각각의 벽시계 시간(`wall_s`), 큐 대기 시간(`queue_s`), 실행 시간(`exec_s`), IBM 하드웨어 사용 시간(`usage_s`).
3. **품질 및 학습 진행** — PADO 증류 회로(회로 123개 × 512 shots)에서 step 1 → step 2 측정 에너지·AR·기울기 코사인 유사도 및 2 step 갱신 후 AR 변화.

- 방식: `pado_safe` (LWPP 사전학습 + `|θ| ≤ 0.3` 증류 후 남은 각도 61개 → step당 회로 `2×61 + 1 = 123`개, 2 step 합계 `246`개 회로).
- 결과는 `results_on_<기기>/extras/qpu_trial/<케이스>/pado_safe_two_step.json`에 쓴다. 본 학습(`train_qpu.ipynb`)이나 1 step 시운전 결과와 섞이지 않는다.
- 이 노트북은 `extras/qpu_trial/` 폴더에서 실행한다. 시작점 파일(`results_on_MI300X/TC3/.../start_points/`)이 없으면 1번 셀에서 자동 생성한다.

## 설정

In [1]:
import datetime
import json
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import rustworkx as rx
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2, Session

from common import device_tag, save_json   # extras/qpu_trial/common.py

# ---- 실행 설정 (보통 여기만 바꾼다)
DRY_RUN = False               # True: 실기기 대신 FAKE_BACKEND 스냅샷으로 로컬 시뮬레이션 (코드 점검용, 사용량 0)
FAKE_BACKEND = "FakeNighthawk" # DRY_RUN용 스냅샷 (qiskit_ibm_runtime.fake_provider)
IBM_BACKEND = "ibm_phoenix"   # 실기기 이름 (configs/TC3.json과 동일하게 "ibm_phoenix")
RUNS = ["pado_safe"]          # PADO 사전학습·증류된 가벼운 회로 (각도 61개 → step당 회로 123개)
STEPS = 2                     # Session 활성화 확인용 연속 2 step 실행
SHOTS = 512                   # 회로(행) 하나당 측정 횟수 (configs/TC3.json 기준 512)
REP_DELAY = None              # 초. None이면 기기 기본값(ibm_phoenix는 1 µs)
RESILIENCE_LEVEL = None       # SamplerV2 직접 비트열 측정 및 후처리 (오차 완화 미적용)
LR = 0.02                     # Adam 학습률 (본 학습과 같음)
SEED_TRANSPILER = 0
TWO_Q_GATE = "cz"             # 배치 점수에 쓸 2큐비트 게이트 (Heron, Nighthawk 계열은 cz)

REPO = Path.cwd().parents[1]          # 이 노트북은 extras/qpu_trial/ 폴더에서 실행한다
CASE = "grid3x3_inst0_seed120490"
START_DIR = REPO / f"results_on_{device_tag()}" / "TC3" / CASE / "start_points"

## API 키 (실기기에서만)

In [3]:
TOKEN = ""     # IBM Cloud API 키. 비우면 이 컴퓨터에 저장된 계정(~/.qiskit/qiskit-ibm.json)을 쓴다. 입력했으면 실행 뒤 지운다
INSTANCE = ""  # 인스턴스 CRN 또는 이름. 비우면 계정 기본값

## backend

In [4]:
if DRY_RUN:
    import importlib
    import warnings
    warnings.filterwarnings("ignore", message="Properties of fake_")
    service = None
    backend = getattr(importlib.import_module("qiskit_ibm_runtime.fake_provider"), FAKE_BACKEND)()
else:
    if TOKEN:
        service = QiskitRuntimeService(channel="ibm_quantum_platform", token=TOKEN, instance=INSTANCE or None)
    else:
        service = QiskitRuntimeService()                     # 저장된 계정
    backend = service.backend(IBM_BACKEND) if IBM_BACKEND else service.least_busy(operational=True, simulator=False)
    try:
        u = service.usage()
        print("usage(초):", {k: u.get(k) for k in ("usage_consumed_seconds", "usage_limit_seconds", "usage_remaining_seconds")})
    except Exception as e:
        print("usage 조회 불가:", e)

cfg = backend.configuration()
print(backend.name, backend.num_qubits, "qubits |", cfg.processor_type, "| 대기 job", backend.status().pending_jobs if not DRY_RUN else 0)
rep = REP_DELAY if REP_DELAY is not None else cfg.default_rep_delay
print(f"rep_delay: 기본 {cfg.default_rep_delay * 1e6:.0f} µs, 허용 {cfg.rep_delay_range[0] * 1e6:.0f}~{cfg.rep_delay_range[1] * 1e6:.0f} µs, 이번 설정 {rep * 1e6:.0f} µs")
if REP_DELAY is not None:
    assert cfg.rep_delay_range[0] <= REP_DELAY <= cfg.rep_delay_range[1], "REP_DELAY가 기기 허용 범위 밖"

OUT = REPO / f"results_on_{backend.name}" / "extras" / "qpu_trial" / CASE
OUT.mkdir(parents=True, exist_ok=True)
print("결과 폴더:", OUT.relative_to(REPO))

qiskit_runtime_service._discover_account:WARNING:2026-10-22 03:02:48,377: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-10-22 03:02:52,666: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (premium, on-prem), the available account instances are: P_QARS_Pusan_univ, QARS_Pusan_univ. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.
qiskit_runtime_service.backends:WARNING:2026-10-22 03:02:52,666: Using instance: P_QARS_Pusan_univ, plan: premium


usage(초): {'usage_consumed_seconds': 2403, 'usage_limit_seconds': 12000, 'usage_remaining_seconds': 9597}
ibm_phoenix 120 qubits | {'family': 'Nighthawk', 'revision': '2'} | 대기 job 107
rep_delay: 기본 1 µs, 허용 0~2000 µs, 이번 설정 1 µs
결과 폴더: results_on_ibm_phoenix/extras/qpu_trial/grid3x3_inst0_seed120490


## 1. 문제, 시작점, 회로 (`pado_safe` 사전학습·증류 회로)

In [5]:
if not (START_DIR / "params.json").exists():
    print("시작점 파일이 없어 quantum/prepare_start_points.py를 먼저 실행합니다...")
    subprocess.run([sys.executable, str(REPO / "quantum" / "prepare_start_points.py"), "--config", str(REPO / "configs" / "TC3.json")], check=True)

# 시작점 파일 두 개: circuit.json(게이트 목록), params.json(θ 값, brute force E_min/E_max, 무노이즈 에너지)
circ_json = json.loads((START_DIR / "circuit.json").read_text())
start = json.loads((START_DIR / "params.json").read_text())
n, e_min, e_max = circ_json["n_qubits"], start["e_min"], start["e_max"]
ar = lambda e: (e_max - e) / (e_max - e_min)   # approximation ratio: E_max이면 0, 바닥 상태 E_min이면 1

# 방식별 회로와 시작 θ. pado_safe(PADO): 증류 뒤 남은 게이트(61개)와 사전학습·증류를 거친 θ
gates = {"standard": circ_json["gates"], "pado_safe": circ_json["pruned_gates"]}
theta0 = {"standard": np.array(start["theta_init"]), "pado_safe": np.array(start["theta_pado_safe_start"])}
e_start = {"standard": start["exact_energy"]["standard_start"], "pado_safe": start["exact_energy"]["pado_safe_start"]}

# 관측량 H = Σ J_ij Z_i Z_j + Σ h_i Z_i (전부 Z 기저 대각 항이므로 2^n개 비트열의 에너지를 미리 표로 만든다)
H = SparsePauliOp.from_sparse_list([("ZZ", [u, v], w) for u, v, w in circ_json["edges"]] +
                                   [("Z", [q], h) for q, h in circ_json["fields"]], num_qubits=n)
_idx = np.arange(1 << n, dtype=np.uint16)
_sz = np.array([1.0 - 2.0 * ((_idx >> q) & 1) for q in range(n)])   # 상태 인덱스 비트 q = (idx >> q) & 1: |0>→+1, |1>→−1
H_diag = np.zeros(1 << n)
for u, v, w in circ_json["edges"]:
    H_diag += w * _sz[u] * _sz[v]
for q, h in circ_json["fields"]:
    H_diag += h * _sz[q]
def sampler_evs_stds(pub_res):                                      # SamplerV2 결과(BitArray) → 행별 평균 에너지와 표준오차
    ba = pub_res.data.meas
    ints = (ba.array[..., 0].astype(np.uint16) << 8) | ba.array[..., 1].astype(np.uint16)
    e_shots = H_diag[ints]
    return e_shots.mean(axis=-1), e_shots.std(axis=-1, ddof=1) / np.sqrt(ba.num_shots)

circuits = {}
for run in RUNS:
    theta = ParameterVector("theta", len(theta0[run]))   # theta[i]의 번호 i = θ 배열의 자리
    qc = QuantumCircuit(n)
    for g in gates[run]:
        if g["gate"] == "H":
            qc.h(g["wires"][0])
        elif g["gate"] == "RZZ":                         # exp(-i θ ZZ / 2)
            qc.rzz(theta[g["param"]], *g["wires"])
        elif g["gate"] == "RZ":                          # exp(-i θ Z / 2)
            qc.rz(theta[g["param"]], g["wires"][0])
        elif g["gate"] == "RX":                          # exp(-i θ X / 2)
            qc.rx(theta[g["param"]], g["wires"][0])
    circuits[run] = qc
    # 확인: 노이즈 없는 Qiskit 에너지 == 시작점 파일의 에너지(PennyLane으로 계산)
    e = float(StatevectorEstimator().run([(qc, H, theta0[run])]).result()[0].data.evs)
    assert abs(e - e_start[run]) < 1e-8, (run, e, e_start[run])
    print(f"{run:10s} angles={len(theta0[run]):3d}  circuits/step={2 * len(theta0[run]) + 1:3d}  "
          f"2-step total circuits={2 * (2 * len(theta0[run]) + 1):3d}  noiseless start AR={ar(e):.4f}")
print(f"brute force: E_min={e_min:.6f}, E_max={e_max:.6f}")

시작점 파일이 없어 quantum/prepare_start_points.py를 먼저 실행합니다...
[TC3 grid3x3_inst0_seed120490 start_points] 1/3 LWPP 사전학습: max_weight 3, 100 step, float32


zero-filter:   0%|          | 0/123 [00:00<?, ?it/s]

[PPS Info] Propagation complete. Terms generated: 2,388


zero-filter: 100%|██████████| 123/123 [00:00<00:00, 219.43it/s, step=0 rows=33 cols=29 nnz=4]    


[PPS Info] Terms retained after pruning: 120 (5.025126% of peak)
[TC3 grid3x3_inst0_seed120490 start_points] compile 1.9s terms=120  100 step 4.6s  E_lwpp -0.1773 -> -10.3274
[TC3 grid3x3_inst0_seed120490 start_points] 2/3 증류: 남은 각도 61/114  (step당 회로: standard 229, PADO 123)
[TC3 grid3x3_inst0_seed120490 start_points] 3/3 노이즈 없는 AR: standard 시작 0.5291 | LWPP 직후 0.9210 | PADO 시작(증류 후) 0.9311
[TC3 grid3x3_inst0_seed120490 start_points] 끝: 저장 results_on_MI300X/TC3/grid3x3_inst0_seed120490/start_points
pado_safe  angles= 61  circuits/step=123  2-step total circuits=246  noiseless start AR=0.9311
brute force: E_min=-8.656307, E_max=9.354969


## 2. 큐비트 배치와 transpile

기기 연결 그래프 안에 3x3 격자(최근접 이웃 12쌍)가 그대로 들어가는 배치가 있으면 그중 오류 점수(2큐비트 게이트 오류 + 측정 오류의 합)가 가장 작은 것을 `initial_layout`으로 고정한다.

In [6]:
# 논리 3x3 격자의 최근접 이웃 연결
grid = rx.PyGraph()
grid.add_nodes_from(range(n))
for i in range(3):
    for j in range(3):
        if j + 1 < 3:
            grid.add_edge(3 * i + j, 3 * i + j + 1, None)       # 오른쪽 이웃
        if i + 1 < 3:
            grid.add_edge(3 * i + j, 3 * (i + 1) + j, None)     # 아래쪽 이웃

coupling = backend.coupling_map.graph.to_undirected(multigraph=False)
twoq_err = backend.target[TWO_Q_GATE]
candidates = []
for mapping in rx.vf2_mapping(coupling, grid, subgraph=True, induced=False):   # {물리 큐비트: 논리 큐비트}
    layout = [0] * n
    for phys, logical in mapping.items():
        layout[logical] = phys
    score = 0.0
    for u, v in grid.edge_list():
        props = twoq_err.get((layout[u], layout[v])) or twoq_err.get((layout[v], layout[u]))
        score += props.error if props is not None and props.error is not None else 1.0
    for q in range(n):
        props = backend.target["measure"].get((layout[q],))
        score += props.error if props is not None and props.error is not None else 1.0
    candidates.append((score, layout))

if candidates:
    score, LAYOUT = min(candidates)
    print(f"3x3 격자 배치 {len(candidates)}개 중 오류 점수가 가장 작은 배치 (점수 {score:.4f}): {LAYOUT}")
else:
    LAYOUT = None
    print(f"{backend.name}에는 3x3 격자가 그대로 들어가는 배치가 없다 (heavy-hex). transpiler 자동 배치를 쓴다.")

pm = generate_preset_pass_manager(optimization_level=3, backend=backend, initial_layout=LAYOUT, seed_transpiler=SEED_TRANSPILER)
isa, phys = {}, {}
for run in RUNS:
    qc_m = circuits[run].copy()
    qc_m.measure_all()                                    # SamplerV2용 Z 기저 측정 추가
    isa[run] = pm.run(qc_m)                               # 기기용 회로 (파라미터는 비어 있음)
    phys[run] = isa[run].layout.initial_index_layout(filter_ancillas=True)
    if LAYOUT is not None:
        assert phys[run] == LAYOUT, (run, phys[run], LAYOUT)
    two_q = sum(1 for inst in isa[run].data if inst.operation.num_qubits == 2)
    dur = isa[run].estimate_duration(backend.target, unit="s")
    print(f"{run:10s} 2-qubit gates={two_q:3d}  depth={isa[run].depth():3d}  circuit duration={dur * 1e6:.1f} µs  "
          f"physical qubits={phys[run]}")
    print(f"           1 step shot 시간 하한 ≈ {2 * len(theta0[run]) + 1} 회로 × {SHOTS} shots × {(rep + dur) * 1e6:.0f} µs"
          f" = {(2 * len(theta0[run]) + 1) * SHOTS * (rep + dur):.1f}초")

3x3 격자 배치 640개 중 오류 점수가 가장 작은 배치 (점수 0.0605): [81, 82, 83, 91, 92, 93, 101, 102, 103]
pado_safe  2-qubit gates= 32  depth= 73  circuit duration=3.9 µs  physical qubits=[81, 82, 83, 91, 92, 93, 101, 102, 103]
           1 step shot 시간 하한 ≈ 123 회로 × 512 shots × 5 µs = 0.3초


## 3. `Session` + `SamplerV2` 설정 및 2 step 연속 실행 (`quantum/train_qpu.ipynb` 방식)

`quantum/train_qpu.ipynb`의 `4-1`, `4-4`, `4-5` 셀과 동일한 세션 구조로 동작한다.
1. **3-1 (`Session` 및 `SamplerV2` 생성)**: `Session(backend=backend)`을 열고 `SamplerV2(mode=session)`을 설정한다.
2. **3-2 (2 step 연속 실행 루프)**:
   - **Step 1 (`t=0`)**: `θ0`의 parameter-shift 회로 123개를 제출하고 `QUEUED → RUNNING → DONE` 상태 변화를 10초 간격으로 찍는다. 이 첫 job이 `RUNNING`에 진입하면 세션이 활성화된다.
   - **Adam 갱신 및 Step 2 즉시 제출 (`t=1`)**: Step 1 결과가 오자마자 0.1초 안에 기울기 계산 및 Adam 업데이트(`θ0 → θ1`)를 수행하고 곧바로 **Step 2 (`t=1`, 회로 123개)**를 제출한다. 세션이 활성화된 상태이므로 Step 2가 대기열(`QUEUED`) 대기 없이 곧바로 실행되는지 확인한다.
   - 2 step이 모두 끝나면 세션을 닫고(`session.close()`) 결과를 저장한다.
3. **3-3 (상태 확인 / 비상 취소 및 세션 종료)**: 실행을 중단(Interrupt)했을 때 현재 job과 세션을 확인·취소할 수 있는 비상용 셀이다.

In [7]:
# [3-1] Session 및 SamplerV2 설정 (quantum/train_qpu.ipynb 4-1 셀과 동일)
session = Session(backend=backend) if not DRY_RUN else None   # 실기기: 첫 job 이후 대기열 없이 연속 실행되는 Session 모드
sampler = SamplerV2(mode=session if session is not None else backend)
sampler.options.default_shots = SHOTS        # 회로(행) 하나당 측정 횟수
if not DRY_RUN and REP_DELAY is not None:
    sampler.options.execution.rep_delay = REP_DELAY   # shot 사이 대기 (기기 허용 범위 안)
if session is not None:
    det = session.details() or {}
    print(f"Session 생성 완료: id={session.session_id} | max_time={det.get('max_time')}s | interactive_timeout={det.get('interactive_timeout')}s")

Session 생성 완료: id=3de3b334-db74-40de-b4e8-2976cbc4bc88 | max_time=57600s | interactive_timeout=60s


In [10]:
# [3-2] PADO (pado_safe) 2 step 연속 실행 루프 (Session 내부에서 step 1 완료 즉시 step 2 제출)
current_job = None
results = {}

for run in RUNS:
    theta = theta0[run].copy()
    k = len(theta)
    m = np.zeros(k)
    v = np.zeros(k)
    step_records = []
    thetas_history = [theta.copy()]          # [theta_0, theta_1, theta_2]

    print(f"=== [{datetime.datetime.now().strftime('%H:%M:%S')}] [{run} Session 2-step 시운전 시작] "
          f"총 {STEPS} step (step당 회로 {2 * k + 1}개 × {SHOTS} shots) ===", flush=True)

    for t in range(STEPS):
        stage = f"{run} step {t + 1}/{STEPS} (t={t})"
        rows = [theta.copy()]                                # 행 0: θ 그대로 → E(θ)
        for i in range(k):
            plus = theta.copy()
            plus[i] += np.pi / 2                             # 행 2i+1: θ_i만 +π/2 → E_i+
            minus = theta.copy()
            minus[i] -= np.pi / 2                            # 행 2i+2: θ_i만 −π/2 → E_i−
            rows += [plus, minus]
        rows = np.array(rows)                                # (2k+1) × k

        t0 = time.perf_counter()
        now_str = datetime.datetime.now().strftime("%H:%M:%S")
        job = sampler.run([(isa[run], rows)], shots=SHOTS)
        current_job = job
        job_id = job.job_id()
        print(f"[{now_str}] [{stage}] job 제출 완료: job_id={job_id} (회로 {len(rows)}개 × {SHOTS} shots)", flush=True)

        # 실시간 상태(QUEUED → RUNNING → DONE) 모니터링 (10초마다 같은 줄 덮어쓰기, 상태 변경 시 줄바꿈)
        if not DRY_RUN:
            last_st = None
            while True:
                st = job.status()
                st_name = st.name if hasattr(st, "name") else str(st)
                if last_st is not None and st_name != last_st:
                    print(flush=True)
                last_st = st_name
                msg = f"  -> [{datetime.datetime.now().strftime('%H:%M:%S')}] [{stage}] status={st_name} (제출 후 경과 {time.perf_counter() - t0:.1f}s)"
                if st_name in ("DONE", "CANCELLED", "ERROR"):
                    print(f"\r{msg}", flush=True)
                    break
                print(f"\r{msg}", end="", flush=True)
                time.sleep(10)

        res = job.result()[0]
        wall_s = time.perf_counter() - t0
        evs, stds = sampler_evs_stds(res)
        energy = float(evs[0])
        grad = (evs[1::2] - evs[2::2]) / 2

        # 노이즈 없는 기준값 (제출 대기를 막기 위해 결과 수신 후 계산)
        evs_ref = np.asarray(StatevectorEstimator().run([(circuits[run], H, rows)]).result()[0].data.evs)
        grad_ref = (evs_ref[1::2] - evs_ref[2::2]) / 2
        cos = float(grad @ grad_ref / (np.linalg.norm(grad) * np.linalg.norm(grad_ref)))

        # IBM이 집계한 사용 시간(초)과 시각(큐 대기 시간 / 실행 시간 분리)
        usage_s, queue_s, exec_s, timestamps = None, None, None, {}
        try:
            usage_s = float(job.usage())
        except Exception:
            pass
        try:
            ts_raw = job.metrics().get("timestamps", {})
            timestamps = {kk: str(vv) for kk, vv in ts_raw.items()}
            to_dt = lambda x: x if hasattr(x, "timestamp") else datetime.datetime.fromisoformat(str(x).replace("Z", "+00:00"))
            if "created" in ts_raw and "running" in ts_raw:
                queue_s = (to_dt(ts_raw["running"]) - to_dt(ts_raw["created"])).total_seconds()
            if "running" in ts_raw and "finished" in ts_raw:
                exec_s = (to_dt(ts_raw["finished"]) - to_dt(ts_raw["running"])).total_seconds()
        except Exception:
            pass

        # Adam 업데이트 (quantum/train_qpu.ipynb 4-4 셀과 동일)
        step_num = t + 1
        m = 0.9 * m + 0.1 * grad
        v = 0.999 * v + 0.001 * (grad ** 2)
        m_hat = m / (1 - 0.9 ** step_num)
        v_hat = v / (1 - 0.999 ** step_num)
        theta_next = theta - LR * m_hat / (np.sqrt(v_hat) + 1e-8)
        e_next_noiseless = float(StatevectorEstimator().run([(circuits[run], H, theta_next)]).result()[0].data.evs)

        rec_step = {
            "t": t, "step": step_num, "job_id": job_id, "n_circuits": int(len(rows)),
            "wall_s": wall_s, "queue_s": queue_s, "exec_s": exec_s, "usage_s": usage_s, "timestamps": timestamps,
            "energy": energy, "energy_std": float(stds[0]), "ar": ar(energy),
            "energy_noiseless": float(evs_ref[0]), "ar_noiseless": ar(float(evs_ref[0])),
            "grad": grad.tolist(), "grad_noiseless": grad_ref.tolist(), "grad_cosine": cos,
            "grad_norm_ratio": float(np.linalg.norm(grad) / np.linalg.norm(grad_ref)),
            "ar_after_step_noiseless": ar(e_next_noiseless),
            "theta_in": theta.tolist(), "theta_out": theta_next.tolist(),
        }
        step_records.append(rec_step)
        theta = theta_next
        thetas_history.append(theta.copy())

        qs = f"{queue_s:.1f}s" if queue_s is not None else "-"
        es = f"{exec_s:.1f}s" if exec_s is not None else "-"
        print(f"  [완료 | {stage} 끝 ({datetime.datetime.now().strftime('%H:%M:%S')})] "
              f"벽시계 {wall_s:.1f}s (큐 대기 {qs} + 실행 {es}) | usage {usage_s}s | "
              f"AR={rec_step['ar']:.4f} (noiseless {rec_step['ar_noiseless']:.4f}) | "
              f"grad cos={cos:.3f} → 갱신 후 noiseless AR={rec_step['ar_after_step_noiseless']:.4f}", flush=True)

    results[run] = {
        "run": run, "backend": backend.name, "dry_run": DRY_RUN,
        "session_id": session.session_id if session is not None else None,
        "shots": SHOTS, "steps": STEPS, "rep_delay_s": rep, "resilience_level": None,
        "physical_qubits": phys[run], "n_params": k, "circuits_per_step": 2 * k + 1,
        "step_records": step_records,
        "theta0": thetas_history[0].tolist(),
        "theta1": thetas_history[1].tolist(),
        "theta2": thetas_history[2].tolist(),
    }
    save_json(OUT / f"{run}_two_step.json", results[run])

if session is not None:
    session.close()
    print("Session closed.")

=== [03:03:11] [pado_safe Session 2-step 시운전 시작] 총 2 step (step당 회로 123개 × 512 shots) ===
[03:03:11] [pado_safe step 1/2 (t=0)] job 제출 완료: job_id=dbcnprs24mrc73ck10r0 (회로 123개 × 512 shots)
  -> [04:58:07] [pado_safe step 1/2 (t=0)] status=QUEUED (제출 후 경과 6895.8s)

KeyboardInterrupt: 

In [ ]:
# [3-3] 상태 확인 및 비상 취소(Cancel) / Session 종료 셀 (필요할 때만 실행; train_qpu.ipynb 4-5 셀과 동일)
if "current_job" in globals() and current_job is not None:
    print("최근 job:", current_job.job_id(), "| status:", current_job.status())
if "session" in globals() and session is not None:
    print("session_id:", session.session_id, "| status:", session.status())

# ---- [비상용] 실행/대기 중인 job을 취소하고 세션을 닫으려면 아래 주석을 풀고 실행하세요:
# if current_job is not None: current_job.cancel(); print("job cancel 요청 완료:", current_job.job_id())
# if session is not None: session.close(); print("session close 완료")
# (특정 job_id 문자열로 직접 취소할 때: service.job("여기에_JOB_ID_입력").cancel())

job cancel 요청 완료: dbcnprs24mrc73ck10r0
session close 완료


## 4. 2 step 전후의 에너지와 비트열 분포 (`θ0` → `θ1` → `θ2`, 로컬 Qiskit Statevector 샘플링)

실기기 job을 추가로 쓰지 않고, 로컬 Qiskit `Statevector`로 `θ0`(시작), `θ1`(1 step 후), `θ2`(2 step 후)의 비트열 표본(`SHOTS`개)과 정확한 확률 분포를 계산해 비교한다.

In [ ]:
from qiskit.quantum_info import Statevector
import matplotlib.pyplot as plt

prob_path = START_DIR.parent / "problem.json"
if not prob_path.exists():
    prob_path = REPO / "problems" / f"{CASE}.json"
prob = json.loads(prob_path.read_text())
ground_str = "".join(str(b) for b in prob["ground_state_bits"])
U = np.array([u for u, v, w in circ_json["edges"]]); V = np.array([v for u, v, w in circ_json["edges"]])
W = np.array([w for u, v, w in circ_json["edges"]]); Q = np.array([q for q, h in circ_json["fields"]]); HF = np.array([h for q, h in circ_json["fields"]])
def energy_of_bits(bits):
    sz = 1 - 2 * bits
    return (sz[:, U] * sz[:, V]) @ W + sz[:, Q] @ HF
all_bits = (np.arange(2 ** n)[:, None] >> np.arange(n)) & 1
E_all = energy_of_bits(all_bits)
bins = np.linspace(e_min, e_max, 41)

for run in RUNS:
    d = results[run]
    th0, th1, th2 = np.array(d["theta0"]), np.array(d["theta1"]), np.array(d["theta2"])
    e0_meas = d["step_records"][0]["energy"]
    e1_meas = d["step_records"][1]["energy"]
    e2_noiseless = float(StatevectorEstimator().run([(circuits[run], H, th2)]).result()[0].data.evs)

    cmp = {}
    stages = (("step0_start", th0, e0_meas, "before: theta_0 (step 1 QPU E)"),
              ("step1_after", th1, e1_meas, "after 1 step: theta_1 (step 2 QPU E)"),
              ("step2_after", th2, e2_noiseless, "after 2 steps: theta_2 (noiseless E)"))
    for tag, th, e_est, _ in stages:
        sv = Statevector(circuits[run].assign_parameters(th))
        sv.seed(SEED_TRANSPILER)
        counts = sv.sample_counts(SHOTS)
        keys = list(counts)
        bits = np.array([[int(ch) for ch in key[::-1]] for key in keys])
        cnt = np.array([counts[k] for k in keys], dtype=float)
        E_s = energy_of_bits(bits)
        p_exact = sv.probabilities()
        cmp[tag] = {
            "energy_estimator": e_est, "ar_estimator": ar(e_est),
            "energy_sampled_mean": float(E_s @ cnt / cnt.sum()),
            "energy_sampled_std": float(np.sqrt(((E_s - E_s @ cnt / cnt.sum()) ** 2) @ cnt / cnt.sum())),
            "ar_sampled": ar(float(E_s @ cnt / cnt.sum())), "best_energy_sampled": float(E_s.min()),
            "p_ground_sampled": counts.get(ground_str[::-1], 0) / cnt.sum(),
            "energy_noiseless": float(E_all @ p_exact), "ar_noiseless": ar(float(E_all @ p_exact)),
            "p_ground_noiseless": float(p_exact[int(ground_str[::-1], 2)]),
            "hist_sampled": np.histogram(E_s, bins=bins, weights=cnt)[0].tolist(),
            "hist_noiseless": (np.histogram(E_all, bins=bins, weights=p_exact)[0] * cnt.sum()).tolist(),
        }
    d["before_after"] = {"mode": "qiskit_statevector_sample", "bins": bins.tolist(), **cmp}
    save_json(OUT / f"{run}_two_step.json", d)

    fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
    for ax, (tag, _, _, title) in zip(axes, stages):
        c = cmp[tag]
        ax.bar(bins[:-1], c["hist_sampled"], width=np.diff(bins), align="edge", color="tab:red", alpha=0.6, label=f"Qiskit sampled ({SHOTS} shots)")
        ax.step(bins[:-1], c["hist_noiseless"], where="post", color="tab:blue", linewidth=1.2, label="noiseless (exact probabilities)")
        ax.axvline(e_min, color="black", linestyle="--", linewidth=1, label="brute-force E_min")
        ax.axvline(c["energy_estimator"], color="tab:red", linestyle=":", linewidth=1.5, label="QPU / ref energy")
        ax.axvline(c["energy_noiseless"], color="tab:blue", linestyle=":", linewidth=1.5, label="noiseless energy")
        ax.set_title(f"{run} {title}\nAR {c['ar_estimator']:.3f} / noiseless {c['ar_noiseless']:.3f}", fontsize=8)
        ax.set_xlabel("energy of sampled bitstring")
    axes[0].set_ylabel(f"shots (of {SHOTS})")
    axes[0].legend(fontsize=6)
    fig.tight_layout()
    fig.savefig(OUT / f"{run}_two_step_before_after.png", dpi=150)
    plt.show()
    for tag, _, _, _ in stages:
        c = cmp[tag]
        print(f"[{run}] {tag:11s} E: QPU/ref {c['energy_estimator']:+.4f} | sampled mean {c['energy_sampled_mean']:+.4f} | noiseless {c['energy_noiseless']:+.4f}"
              f" || P(ground): sampled {c['p_ground_sampled']:.3f} vs noiseless {c['p_ground_noiseless']:.3f}")

## 5. Step 1 vs Step 2 `Session` 대기 시간 비교 및 요약

In [ ]:
print(f"{'방식':10s} {'step':>4s} {'회로':>5s} {'큐대기(초)':>10s} {'실행(초)':>9s} {'총벽시계':>8s} {'usage(초)':>10s} {'AR 측정':>8s} {'AR 무노이즈':>10s} {'grad cos':>9s} {'갱신뒤 AR':>10s}")
for run, d in results.items():
    for s in d["step_records"]:
        qs = f"{s['queue_s']:.1f}" if s.get("queue_s") is not None else "-"
        es = f"{s['exec_s']:.1f}" if s.get("exec_s") is not None else "-"
        us = f"{s['usage_s']:.1f}" if s.get("usage_s") is not None else "-"
        print(f"{run:10s} {s['step']:4d} {s['n_circuits']:5d} {qs:>10s} {es:>9s} {s['wall_s']:8.1f} {us:>10s} "
              f"{s['ar']:8.4f} {s['ar_noiseless']:10.4f} {s['grad_cosine']:9.3f} {s['ar_after_step_noiseless']:10.4f}")
    if len(d["step_records"]) >= 2:
        q1, q2 = d["step_records"][0].get("queue_s"), d["step_records"][1].get("queue_s")
        u_tot = sum(s.get("usage_s") or 0.0 for s in d["step_records"])
        print(f"\n[Session 효과 확인] session_id={d.get('session_id')}")
        print(f"  - Step 1 큐 대기 시간: {f'{q1:.1f}초' if q1 is not None else '-'} (첫 job 큐 진입 대기)")
        print(f"  - Step 2 큐 대기 시간: {f'{q2:.1f}초' if q2 is not None else '-'} (Session 활성화 후 연속 실행)")
        print(f"  - 2 step 합계 하드웨어 usage: {u_tot:.1f}초")

if service is not None:
    try:
        u = service.usage()
        print(f"계정 한도 {u.get('usage_limit_seconds')}초, 남은 {u.get('usage_remaining_seconds')}초")
    except Exception:
        pass
save_json(OUT / "summary_two_step.json", {"backend": backend.name, "shots": SHOTS, "rep_delay_s": rep, "runs": results})
print("saved", (OUT / "pado_safe_two_step.json").relative_to(REPO))